# 01 - Attention U-Net segmentation scores on REAL MuralDH damage
Evaluates the **Attention U-Net** on the real, human-annotated MuralDH `Mural_seg` images.

**Protocol (no test-set tuning):** the 760 real training images are split 608 / 152 (seed 42). The damage threshold is chosen on the 152 validation images; Dice / IoU / precision / recall are reported once on the 201 untouched test images.

**Kaggle setup**
1. **Add Input -> Datasets:** `MuralDH`.
2. **Add Input -> Your Work -> Notebooks:** your `Attention_Unet` notebook (its output holds the `.keras` file). Or upload the `.keras` file as a dataset and add that.
3. Accelerator: GPU T4 (CPU also works, just slower). Run All.
4. Download from `/kaggle/working/results/`: `segmentation_eval.json`, `thresholds.json`, `threshold_sweep.png`, `segmentation_qualitative.png`.


In [ ]:
import os, json, glob, random
import numpy as np, cv2
import matplotlib.pyplot as plt
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
print("tensorflow", tf.__version__)

# ---- find Mural_seg (train/images, train/labels, test/images, test/labels) anywhere in the inputs
seg_roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/Mural_seg/**/train/images", recursive=True)]
seg_roots += [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/train/images", recursive=True) if "Mural_seg" in p]
assert seg_roots, "Mural_seg not found - add the MuralDH dataset as input"
DATA = seg_roots[0]
TRAIN_IMG, TRAIN_LAB = f"{DATA}/train/images", f"{DATA}/train/labels"
TEST_IMG,  TEST_LAB  = f"{DATA}/test/images",  f"{DATA}/test/labels"
print("data:", DATA, "|", len(os.listdir(TRAIN_IMG)), "train images,", len(os.listdir(TEST_IMG)), "test images")

# ---- find the Attention U-Net weights (.keras / .h5) in the inputs
cands = glob.glob("/kaggle/input/**/*.keras", recursive=True) + glob.glob("/kaggle/input/**/*.h5", recursive=True)
print("model files found:", cands)
att = [p for p in cands if "att" in os.path.basename(p).lower()]
MODEL_PATH = (att or [p for p in cands if "muralv2" not in p])[0] if cands else None
assert MODEL_PATH, "No .keras/.h5 file found - add the Attention_Unet notebook output (or a dataset with the file) as input"
print("USING:", MODEL_PATH)
os.makedirs("/kaggle/working/results", exist_ok=True)


In [ ]:
def load_any(path):
    for kw in (dict(compile=False, safe_mode=False), dict(compile=False)):
        try: return tf.keras.models.load_model(path, **kw)
        except TypeError: continue
    raise RuntimeError("could not load " + path)
model = load_any(MODEL_PATH)
print("input", model.input_shape, "output", model.output_shape)
loaded = {"attention_unet": model}
MODELS = {"attention_unet": MODEL_PATH}

def prep(rgb, size, channels):
    x = cv2.resize(rgb, size, interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
    if channels == 4:   # RGB + Canny 100/200, as in the Attention U-Net training notebook
        g = cv2.cvtColor((x * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        x = np.concatenate([x, cv2.Canny(g, 100, 200)[..., None].astype(np.float32) / 255.0], -1)
    return x

def load_pair(img_dir, lab_dir, fname):
    rgb = cv2.cvtColor(cv2.imread(os.path.join(img_dir, fname)), cv2.COLOR_BGR2RGB)
    stem = os.path.splitext(fname)[0]
    lp = os.path.join(lab_dir, fname)
    if not os.path.exists(lp):
        lp = next(iter(glob.glob(os.path.join(lab_dir, stem + ".*"))), lp)
    lab = cv2.imread(lp, cv2.IMREAD_GRAYSCALE)
    return rgb, (lab > 0)            # works for 0/1 and 0/255 label files

def predict_probs(model, rgb_list, bs=8):
    _, h, w, c = model.input_shape
    X = np.stack([prep(r, (w, h), c) for r in rgb_list])
    P = model.predict(X, batch_size=bs, verbose=0)
    P = P[..., 0] if P.ndim == 4 else P
    return [cv2.resize(p, (r.shape[1], r.shape[0]), interpolation=cv2.INTER_LINEAR) for p, r in zip(P, rgb_list)]


In [ ]:
def metrics(pred, gt):
    tp = float((pred & gt).sum()); fp = float((pred & ~gt).sum()); fn = float((~pred & gt).sum())
    dice = 1.0 if (tp+fp+fn)==0 else 2*tp/(2*tp+fp+fn)
    iou  = 1.0 if (tp+fp+fn)==0 else tp/(tp+fp+fn)
    prec = 1.0 if (tp+fp)==0 else tp/(tp+fp)
    rec  = 1.0 if (tp+fn)==0 else tp/(tp+fn)
    return dict(dice=dice, iou=iou, precision=prec, recall=rec, tp=tp, fp=fp, fn=fn)

def collect(model, img_dir, lab_dir, files):
    rgbs, gts = zip(*[load_pair(img_dir, lab_dir, f) for f in files])
    return predict_probs(model, list(rgbs)), list(gts)

def sweep(probs, gts, thresholds):
    rows = []
    for t in thresholds:
        ms = [metrics(p >= t, g) for p, g in zip(probs, gts)]
        macro = {k: float(np.mean([m[k] for m in ms])) for k in ("dice","iou","precision","recall")}
        tp, fp, fn = (sum(m[k] for m in ms) for k in ("tp","fp","fn"))
        micro = dict(dice_micro=2*tp/max(2*tp+fp+fn,1), iou_micro=tp/max(tp+fp+fn,1))
        rows.append(dict(threshold=round(float(t),2), **macro, **micro))
    return rows

# --- validation split (same 608/152 protocol as the review report; fixed seed)
all_train = sorted(os.listdir(TRAIN_IMG)); rng = random.Random(42); rng.shuffle(all_train)
val_files = all_train[608:]; test_files = sorted(os.listdir(TEST_IMG))
print("validation:", len(val_files), "| test:", len(test_files))
THRS = np.round(np.arange(0.05, 0.96, 0.05), 2)

In [ ]:
results, best_thr, per_image = {}, {}, {}
for key, model in loaded.items():
    vp, vg = collect(model, TRAIN_IMG, TRAIN_LAB, val_files)
    val_rows = sweep(vp, vg, THRS)
    best = max(val_rows, key=lambda r: r["dice"])                 # chosen on VALIDATION only
    best_thr[key] = best["threshold"]
    tp_, tg_ = collect(model, TEST_IMG, TEST_LAB, test_files)
    test_rows = sweep(tp_, tg_, THRS)
    at_best = next(r for r in test_rows if r["threshold"] == best["threshold"])
    at_05   = next(r for r in test_rows if r["threshold"] == 0.5)
    pi = [dict(image=f, **{k: v for k, v in metrics(p >= best["threshold"], g).items() if k in ("dice","iou","precision","recall")},
               gt_pct=float(g.mean()*100), pred_pct=float((p >= best["threshold"]).mean()*100))
          for f, p, g in zip(test_files, tp_, tg_)]
    per_image[key] = pi
    results[key] = dict(model_file=os.path.basename(MODELS[key]), input_shape=list(model.input_shape[1:]),
                        validation_images=len(val_files), test_images=len(test_files),
                        selected_threshold=best["threshold"], validation_at_selected=best,
                        test_at_selected=at_best, test_at_0_5=at_05, validation_sweep=val_rows, test_sweep=test_rows,
                        test_dice_std=float(np.std([r["dice"] for r in pi])))
    print(f"[Attention U-Net] threshold chosen on val = {best['threshold']}  ->  TEST Dice {at_best['dice']:.3f}  IoU {at_best['iou']:.3f}  "
          f"P {at_best['precision']:.3f}  R {at_best['recall']:.3f}   (at 0.5: Dice {at_05['dice']:.3f})")
json.dump(results, open("/kaggle/working/results/segmentation_eval.json", "w"), indent=2)
json.dump(best_thr, open("/kaggle/working/results/thresholds.json", "w"), indent=2)
print("saved results/segmentation_eval.json and thresholds.json")

In [ ]:
# threshold curves (validation) + qualitative panel for the thesis figures
fig, axes = plt.subplots(1, len(results), figsize=(6.5*len(results), 4.2), squeeze=False)
for ax, (key, r) in zip(axes[0], results.items()):
    xs = [row["threshold"] for row in r["test_sweep"]]
    for m in ("dice", "iou", "precision", "recall"):
        ax.plot(xs, [row[m] for row in r["test_sweep"]], marker="o", ms=3, label=m)
    ax.axvline(r["selected_threshold"], color="gray", ls="--", label=f"val-selected {r['selected_threshold']}")
    ax.set_title("Attention U-Net: test metrics vs threshold"); ax.set_xlabel("threshold"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/kaggle/working/results/threshold_sweep.png", dpi=160); plt.show()

key = next(iter(loaded)); model = loaded[key]; thr = results[key]["selected_threshold"]
sel = test_files[::max(len(test_files)//6, 1)][:6]
rgbs, gts = zip(*[load_pair(TEST_IMG, TEST_LAB, f) for f in sel]); probs = predict_probs(model, list(rgbs))
fig, axes = plt.subplots(len(sel), 4, figsize=(13, 3.2*len(sel)))
for i, (r, g, p) in enumerate(zip(rgbs, gts, probs)):
    ov = r.copy(); ov[p >= thr] = (0.5*ov[p >= thr] + 0.5*np.array([255, 0, 0])).astype(np.uint8)
    for j, (im, t, cm) in enumerate([(r, "image", None), (g, "ground truth", "gray"), (p >= thr, "prediction", "gray"), (ov, "overlay", None)]):
        axes[i, j].imshow(im, cmap=cm); axes[i, j].set_title(t, fontsize=9); axes[i, j].axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/results/segmentation_qualitative.png", dpi=140); plt.show()

## Next
Download the four files from `/kaggle/working/results/` and send `segmentation_eval.json` (or a screenshot of the `[Attention U-Net] ... TEST Dice ...` line) for the report.
